# Watershed Simple por Slice + Limpieza Manual (v2)

**Pipeline simplificado:**
1. Cargar `.mrc`
2. Erosion ligera 2D (solo limpieza, no separacion agresiva)
3. Watershed por slice, INDEPENDIENTE (sin propagacion de semillas entre slices -- esto se
   anade despues, una vez que la separacion lateral este limpia)
4. Revisar en 3D, corregir manualmente sobre la mascara binaria, re-correr

**Nota sobre los IDs:** en esta version, el ID de un filamento en el slice Z NO es necesariamente
el mismo que en el slice Z+1 -- el enlazado de identidad entre slices se deja para un paso
posterior, una vez que la separacion lateral (dentro de cada slice) este limpia.

## 1. Carga

Imports y definicion de rutas (`input_file`, `output_dir`). La carga real de los datos
(original vs. version editada) se hace en la seccion 1c, mas abajo.

In [10]:
import numpy as np
from scipy import ndimage as ndi
from scipy.ndimage import distance_transform_edt, label as cc_label, gaussian_filter, binary_erosion
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
import napari
import mrcfile

input_file = r"C:\PhD\Actin_ET\Segmentation\Amoeboid\L8_P14\L8_Position_14_actin.mrc"
VOXEL_SIZE_A = 12.4  # Å/voxel -- confirmado isotropico en X/Y/Z

### 1b. Carpeta de salida

Todos los resultados (watershed, mascara editada) se guardan en `C:\PhD\Actin_ET\Bundle Analysis`.

In [11]:
from pathlib import Path

output_dir = Path(r"C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L8_P14")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Carpeta de salida: {output_dir}")

Carpeta de salida: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L8_P14


### 1c. Cargar datos -- elige UNA de las dos celdas siguientes

- **1c-i** carga el `.mrc` ORIGINAL (`input_file`), sin ninguna correccion.
- **1c-ii** carga la version EDITADA mas reciente desde `output_dir` (la ultima vez que guardaste
  correcciones manuales en la seccion 7).

Corre solo una de las dos, segun si estas empezando de cero o retomando una sesion de limpieza
manual anterior.

In [12]:
# 1c-i: cargar el .mrc ORIGINAL
with mrcfile.open(input_file, permissive=True) as mrc:
    seg_raw = mrc.data.copy()
seg_binary = (seg_raw > 0).astype(bool)

print(f"Cargado ORIGINAL: {input_file}")
print(f"Shape (Z,Y,X): {seg_binary.shape}")
print(f"Voxels de actina: {seg_binary.sum():,}")

Cargado ORIGINAL: C:\PhD\Actin_ET\Segmentation\Amoeboid\L8_P14\L8_Position_14_actin.mrc
Shape (Z,Y,X): (300, 1024, 1024)
Voxels de actina: 895,526


In [ ]:
# 1c-ii: cargar la version EDITADA mas reciente desde output_dir
edited_files = sorted(
    output_dir.glob(Path(input_file).stem + "_edited_v*.mrc"),
    key=lambda p: int(p.stem.split("_v")[-1])
)

if not edited_files:
    raise FileNotFoundError(
        f"No se encontro ninguna version editada en {output_dir}. "
        f"Corre la celda 1c-i (original) en su lugar."
    )

latest_edited = edited_files[-1]
with mrcfile.open(str(latest_edited), permissive=True) as mrc:
    seg_raw = mrc.data.copy()
seg_binary = (seg_raw > 0).astype(bool)

print(f"Cargada version EDITADA: {latest_edited}")
print(f"(De {len(edited_files)} version(es) disponibles: {[f.name for f in edited_files]})")
print(f"Shape (Z,Y,X): {seg_binary.shape}")
print(f"Voxels de actina: {seg_binary.sum():,}")

In [11]:
# 1c-ii: cargar la version EDITADA mas reciente desde output_dir
# Busca CUALQUIER archivo "*_edited_v*.mrc" en output_dir, sin exigir que el prefijo
# coincida con el nombre de input_file -- mas simple y robusto (no depende de renombrar
# el original ni de escapar caracteres especiales si el nombre del input tiene corchetes
# u otros simbolos especiales de glob).
edited_files = sorted(
    output_dir.glob("*_edited_v*.mrc"),
    key=lambda p: int(p.stem.split("_v")[-1])
)

if not edited_files:
    raise FileNotFoundError(
        f"No se encontro ninguna version editada (*_edited_v*.mrc) en {output_dir}. "
        f"Corre la celda 1c-i (original) en su lugar."
    )

latest_edited = edited_files[-1]
with mrcfile.open(str(latest_edited), permissive=True) as mrc:
    seg_raw = mrc.data.copy()
seg_binary = (seg_raw > 0).astype(bool)

print(f"Cargada version EDITADA: {latest_edited}")
print(f"(De {len(edited_files)} version(es) disponibles: {[f.name for f in edited_files]})")
print(f"Shape (Z,Y,X): {seg_binary.shape}")
print(f"Voxels de actina: {seg_binary.sum():,}")

Cargada version EDITADA: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\P3\P3_actin_edited_v1.mrc
(De 1 version(es) disponibles: ['P3_actin_edited_v1.mrc'])
Shape (Z,Y,X): (300, 1024, 1024)
Voxels de actina: 241,366


## 2. Erosion ligera + Watershed por slice (independiente)

Erosion 2D ligera SOLO para limpiar el contacto marginal entre filamentos y dar una distance
transform mas nitida -- no para separar del todo (esa funcion la hace watershed). El watershed
final siempre reparte sobre la mascara y distancia ORIGINALES (sin erosionar), asi que no se
pierden voxels en el resultado.

Cada slice Z se procesa de forma completamente independiente: no hay propagacion ni enlazado
con slices vecinos. Esto es deliberado por ahora -- queremos primero que la separacion DENTRO
de cada slice sea buena, sin la complejidad anadida de mantener identidad entre slices.

In [13]:
def watershed_slice(mask_2d, min_distance=4, smooth_sigma=2.5, seed_erosion_radius=1):
    """
    Separa una mascara binaria 2D en filamentos individuales via watershed, en un solo slice,
    sin ninguna dependencia de otros slices.

    0. Erosion ligera 2D (seed_erosion_radius), SOLO para detectar semillas mas limpias -- no
       afecta el resultado final (el watershed del paso 3 siempre usa la mascara original).
    1. Distance transform (sobre la mascara erosionada para detectar semillas; sobre la original
       para el reparto final de watershed).
    2. Suavizado gaussiano de la distance transform de semillas, para evitar multiples maximos
       espurios dentro de un mismo filamento recto (sobre-segmentacion por ruido).
    3. peak_local_max -> semillas. Watershed con esas semillas sobre la mascara/distancia originales.

    Devuelve: labels (int array, 0=fondo, 1..N=filamentos en ESTE slice), seed_coords
    """
    distance = distance_transform_edt(mask_2d)

    if seed_erosion_radius > 0:
        struct_2d = np.ones((2*seed_erosion_radius+1, 2*seed_erosion_radius+1), dtype=bool)
        mask_for_seeds = binary_erosion(mask_2d, structure=struct_2d)
        if mask_for_seeds.sum() == 0:
            mask_for_seeds = mask_2d
        distance_for_seeds_base = distance_transform_edt(mask_for_seeds)
    else:
        distance_for_seeds_base = distance

    distance_for_seeds = (
        gaussian_filter(distance_for_seeds_base, sigma=smooth_sigma) if smooth_sigma > 0 else distance_for_seeds_base
    )

    coords = peak_local_max(distance_for_seeds, min_distance=min_distance, labels=mask_2d)

    seed_mask = np.zeros_like(distance, dtype=bool)
    seed_mask[tuple(coords.T)] = True
    seed_labels, n_seeds = cc_label(seed_mask)

    labels_ws = watershed(-distance, seed_labels, mask=mask_2d)

    return labels_ws, coords, n_seeds


# ====== PARAMETROS ======
MIN_DISTANCE = 4        # voxels -- separacion minima entre semillas
SMOOTH_SIGMA = 2.5      # voxels -- sigma del suavizado de la distance transform (evita romper
                        # filamentos rectos en varios labels por ruido)
SEED_EROSION_RADIUS = 1 # voxels -- erosion ligera 2D solo para deteccion de semillas
# =========================

n_slices = seg_binary.shape[0]
labels_ws_volume = np.zeros_like(seg_binary, dtype=np.int32)
n_filaments_per_slice = np.zeros(n_slices, dtype=int)

for z in range(n_slices):
    slice_mask_z = seg_binary[z]
    if slice_mask_z.sum() == 0:
        continue
    labels_z, _, _ = watershed_slice(
        slice_mask_z, min_distance=MIN_DISTANCE, smooth_sigma=SMOOTH_SIGMA,
        seed_erosion_radius=SEED_EROSION_RADIUS
    )
    labels_ws_volume[z] = labels_z
    n_filaments_per_slice[z] = labels_z.max()

print(f"Procesados {n_slices} slices (independientes, sin enlazar).")
print(f"Filamentos por slice (solo slices no vacios): "
      f"min={n_filaments_per_slice[n_filaments_per_slice>0].min()}, "
      f"median={np.median(n_filaments_per_slice[n_filaments_per_slice>0]):.0f}, "
      f"max={n_filaments_per_slice.max()}")

Procesados 300 slices (independientes, sin enlazar).
Filamentos por slice (solo slices no vacios): min=1, median=88, max=238


## 3. Sanity check: voxels preservados

Watershed no deberia perder voxels (reparte, no elimina) -- a diferencia de la erosion sola.

In [14]:
n_voxels_original = seg_binary.sum()
n_voxels_watershed = (labels_ws_volume > 0).sum()

print(f"Voxels originales:  {n_voxels_original:,}")
print(f"Voxels en watershed: {n_voxels_watershed:,}")
print(f"Diferencia: {n_voxels_original - n_voxels_watershed:,} "
      f"({100*(n_voxels_original - n_voxels_watershed)/n_voxels_original:.2f}%)")

Voxels originales:  895,526
Voxels en watershed: 895,423
Diferencia: 103 (0.01%)


## 4. Iterar parametros

Si ves sobre-segmentacion (un filamento recto partido en varios labels): sube `SMOOTH_SIGMA`.
Si ves sub-segmentacion (2+ filamentos comparten label): baja `SMOOTH_SIGMA` o `MIN_DISTANCE`,
o sube `SEED_EROSION_RADIUS`.

Vuelve a correr la seccion 2 con nuevos valores y revisa en Napari (seccion 5) antes de pasar
a la limpieza manual.

## 5. Visualizar en Napari (3D)

`seg_binary` (mascara original, para referencia) + `labels_ws_volume` (resultado de watershed,
coloreado por label). Recuerda: los colores NO son consistentes entre distintos valores de Z
todavia (eso se resuelve en un paso posterior).

In [15]:
viewer_3d = napari.Viewer(title="Watershed por slice (independiente, sin enlazar)")
viewer_3d.add_labels(seg_binary.astype(np.uint8), name="original (binary)", opacity=0.3)
viewer_3d.add_labels(labels_ws_volume, name="watershed (por slice, sin enlazar)", opacity=0.8)
viewer_3d.dims.ndisplay = 3

print("Viewer en 3D. Rota y revisa si la separacion lateral entre filamentos vecinos es buena.")

Viewer en 3D. Rota y revisa si la separacion lateral entre filamentos vecinos es buena.


## 6. Guardar resultado de watershed

Guarda `labels_ws_volume` como .mrc en la carpeta `processed`, para no tener que re-correr
todo el pipeline si vuelves a este notebook mas tarde. Incluye un sufijo con los parametros
usados, para distinguir entre intentos si pruebas varias combinaciones.

In [16]:
ws_output_path = output_dir / (
    Path(input_file).stem + f"_watershed_md{MIN_DISTANCE}_ss{SMOOTH_SIGMA}_er{SEED_EROSION_RADIUS}.mrc"
)

# .mrc no soporta int32 -- usamos int16 (hasta 32,767 labels distintos, mas que suficiente aqui)
max_label = labels_ws_volume.max()
if max_label > 32767:
    raise ValueError(f"max_label={max_label} excede el rango de int16. Avisa si esto pasa.")

with mrcfile.new(str(ws_output_path), overwrite=True) as mrc_out:
    mrc_out.set_data(labels_ws_volume.astype(np.int16))
    mrc_out.voxel_size = VOXEL_SIZE_A

print(f"Guardado: {ws_output_path}")
print("Nota: los labels NO son consistentes entre slices Z todavia (ver nota en la introduccion).")

Guardado: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L8_P14\L8_Position_14_actin_watershed_md4_ss2.5_er1.mrc
Nota: los labels NO son consistentes entre slices Z todavia (ver nota en la introduccion).


## 7. Correccion manual en 3D + re-run

Identifica visualmente (rotando en 3D) zonas donde el watershed fusiono mal dos filamentos
distintos, y borra esos voxels de contacto directamente sobre `seg_binary` usando el pincel
de Napari en modo 3D (`n edit dim = 3`). Despues, vuelve a correr la seccion 2 completa con
la mascara corregida.

Filamentos rotos sin razon (sin fusion cerca) no se arreglan borrando -- si eso es frecuente
despues de limpiar las fusiones, dimelo y vemos otra herramienta para ese caso especifico.

In [17]:
if "seg_binary_edited" not in dir():
    seg_binary_edited = seg_binary.copy()

viewer_edit = napari.Viewer(title="Correccion manual 3D -- borra voxels en zonas de fusion")
viewer_edit.add_labels(seg_binary_edited.astype(np.uint8), name="seg_binary (editable)", opacity=0.5)
viewer_edit.add_labels(labels_ws_volume, name="watershed result (referencia, no editar)", opacity=0.8)
viewer_edit.dims.ndisplay = 3

print("Selecciona la capa 'seg_binary (editable)', activa el pincel, sube 'n edit dim' a 3,")
print("y borra voxels en las zonas de fusion que veas comparando con la capa de referencia.")

Selecciona la capa 'seg_binary (editable)', activa el pincel, sube 'n edit dim' a 3,
y borra voxels en las zonas de fusion que veas comparando con la capa de referencia.


In [18]:
# Extraer la mascara editada de la capa de Napari de vuelta a numpy
seg_binary_edited = viewer_edit.layers["seg_binary (editable)"].data.astype(bool)

n_before = seg_binary.sum()
n_after = seg_binary_edited.sum()
print(f"Voxels antes: {n_before:,} | despues: {n_after:,} | borrados: {n_before - n_after:,}")

seg_binary_backup = seg_binary.copy()  # por si quieres revertir en memoria
seg_binary = seg_binary_edited.copy()

# Guardar a disco con un contador de version, para no sobrescribir intentos anteriores
existing = list(output_dir.glob(Path(input_file).stem + "_edited_v*.mrc"))
next_version = len(existing) + 1
edited_output_path = output_dir / (Path(input_file).stem + f"_edited_v{next_version}.mrc")

with mrcfile.new(str(edited_output_path), overwrite=True) as mrc_out:
    mrc_out.set_data(seg_binary.astype(np.int8))
    mrc_out.voxel_size = VOXEL_SIZE_A

print(f"\nGuardado: {edited_output_path}")
print("'seg_binary' actualizado en memoria. Vuelve a correr la SECCION 2 completa.")
print("(Version sin editar de esta sesion disponible en 'seg_binary_backup'.)")

Voxels antes: 895,526 | despues: 895,526 | borrados: 0

Guardado: C:\PhD\Actin_ET\Bundle Analysis\Amoeboid\L8_P14\L8_Position_14_actin_edited_v1.mrc
'seg_binary' actualizado en memoria. Vuelve a correr la SECCION 2 completa.
(Version sin editar de esta sesion disponible en 'seg_binary_backup'.)


## Proximos pasos

Una vez que la separacion lateral por slice este limpia (sin fusiones ni roturas obvias):

1. **Enlazado de identidad entre slices** (Z), para que cada filamento tenga un ID consistente
   a lo largo de Y -- necesario para medir distancias/orientacion de forma continua, no solo
   punto a punto dentro de un slice. Esto se hace como paso aparte sobre los datos ya limpios.
2. **Extraccion de centroide/orientacion por filamento** (SVD/PCA).
3. **Calculo de distancias inter-filamento y parallelism** para el bundle-scoring.